In [6]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv("/kaggle/input/hranalytics/WA_Fn-UseC_-HR-Employee-Attrition.csv")

In [ ]:
X = df.drop("Attrition", axis=1)
y = df["Attrition"].map({"Yes": 1, "No": 0})

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

In [ ]:
base_estimators = [
    ("svm", SVC(probability=True, kernel="rbf", C=10, gamma="scale")),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(150, 100, 50),
        max_iter=2000,
        random_state=42
    ))
]

In [ ]:
stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=3000),
    cv=5
)

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", stacking_model)
])

In [ ]:
print("Training SVM + MLP hybrid model ...")
model.fit(X, y)

In [ ]:
n_test = int(0.3 * len(X))
synthetic_X = X.sample(n=n_test, random_state=321).copy()

rng = np.random.default_rng(321)

In [ ]:
for col in numerical_cols:
    noise = rng.normal(0, 0.1, size=len(synthetic_X))
    if np.issubdtype(synthetic_X[col].dtype, np.integer):
        synthetic_X[col] = np.round(synthetic_X[col] + noise).astype(int)
    else:
        synthetic_X[col] = synthetic_X[col] + noise

In [ ]:
for col in categorical_cols:
    unique_vals = X[col].unique()
    mask = rng.random(len(synthetic_X)) < 0.1 
    if len(unique_vals) > 1:
        synthetic_X.loc[mask, col] = rng.choice(unique_vals, size=mask.sum())

In [ ]:
synthetic_y = model.predict(synthetic_X)
synthetic_y_attr = pd.Series(synthetic_y).map({1: "Yes", 0: "No"})

demo_test_df = synthetic_X.copy()
demo_test_df["Attrition"] = synthetic_y_attr.values

In [ ]:
y_pred = model.predict(synthetic_X)

accuracy = accuracy_score(synthetic_y, y_pred)

In [7]:
print("\nSVM + MLP RESULTS")
print("-----------------------")
print("Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(synthetic_y, y_pred))

Training SVM + MLP hybrid model ...

SVM + MLP RESULTS
-----------------------
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       362
           1       1.00      1.00      1.00        79

    accuracy                           1.00       441
   macro avg       1.00      1.00      1.00       441
weighted avg       1.00      1.00      1.00       441

